## 19. IFRS 9 Staging & Expected Credit Loss

IFRS 9 requires "live" (not-yet-resolved) loans to be staged by credit deterioration and provisioned accordingly — Stage 1 loans get a 12-month ECL, Stage 2/3 loans get a lifetime ECL. Week 1's PD/LGD models were deliberately trained only on *resolved* loans (Fully Paid / Charged Off / Default), so this section pulls the original, unfiltered dataset back in to recover the live population staging actually needs, memory-safely (reading only the needed columns, in chunks).

In [ ]:
# ============================================================
# CELL 1 (memory-safe rebuild) — Load ONLY live loans, ONLY needed columns
# ============================================================
import gc

# Free up memory from objects we don't need right now before loading anything new
for var_name in ['long_df', 'long_rows', 'tv_sample', 'sf_stage2']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()

# Only read the columns the model and staging logic actually require --
# this avoids loading 150+ unused columns into memory.
needed_cols = list(set(
    feature_cols + cox_features +
    ['loan_status', 'grade', 'home_ownership', 'pub_rec',
     'loan_amnt', 'annual_inc', 'term', 'int_rate']
))

live_statuses = ['Current', 'Late (16-30 days)', 'Late (31-60 days)',
                  'Late (61-90 days)', 'Late (91-120 days)', 'In Grace Period']

# Read in chunks, filtering to live-status rows only as we go, so the full
# unfiltered file is never held in memory all at once.
chunks = []
chunk_iter = pd.read_csv(
    "lending_club_data/accepted_2007_to_2018Q4.csv.gz",
    compression='gzip', low_memory=False,
    usecols=lambda c: c in needed_cols,
    chunksize=200_000
)

for chunk in chunk_iter:
    live_chunk = chunk[chunk['loan_status'].isin(live_statuses)]
    if len(live_chunk) > 0:
        chunks.append(live_chunk.copy())

df_live = pd.concat(chunks, ignore_index=False)
del chunks
gc.collect()

print(f"Live (still-active) loans recovered: {len(df_live):,}")
print(df_live['loan_status'].value_counts())

In [ ]:
# ============================================================
# CELL 2 — Apply the Week 1 cleaning pipeline to live loans
# ============================================================
df_live_cleaned = df_live.copy()

# WOE mapping -- reuse the exact mapping logic from Week 1.
# If you still have the original WOE map dict/Series in memory under its
# original name (e.g. grade_woe_map), use that directly instead of this
# recomputation. This recomputes from df_cleaned as a safe fallback.
for feature in ['grade', 'home_ownership']:
    if feature in df_live_cleaned.columns:
        woe_lookup = df_cleaned.groupby(df.loc[df_cleaned.index, feature])[f'{feature}_woe'].first()
        df_live_cleaned[f'{feature}_woe'] = df_live_cleaned[feature].map(woe_lookup)

# Same derived features as Week 1
if 'pub_rec' in df_live_cleaned.columns:
    df_live_cleaned['has_pub_rec'] = (df_live_cleaned['pub_rec'] > 0).astype(int)
df_live_cleaned['loan_to_income'] = df_live_cleaned['loan_amnt'] / df_live_cleaned['annual_inc'].clip(lower=1)

# Keep only rows with every required model feature present
df_live_cleaned = df_live_cleaned.dropna(subset=feature_cols)
print(f"Live loans with complete features for scoring: {len(df_live_cleaned):,}")
print(df_live_cleaned['loan_status'].value_counts())

In [ ]:
# ============================================================
# CELL 3 — Score live loans with the calibrated PD model
# ============================================================
current_pd_live = pd.Series(
    calibrated_lgb.predict_proba(df_live_cleaned[feature_cols])[:, 1],
    index=df_live_cleaned.index
)
print("Live-loan current PD distribution:")
print(current_pd_live.describe())

In [ ]:
# ============================================================
# CELL 4 — Real DPD state from actual current loan_status
# ============================================================
dpd_state_labels = {
    0: 'Current', 1: '1-30 DPD', 2: '31-60 DPD',
    3: '61-90 DPD', 4: 'Default', 5: 'Prepaid'
}

df_live_cleaned['current_dpd_state'] = df_live_cleaned['loan_status'].apply(assign_state)
df_live_cleaned['current_dpd_label'] = df_live_cleaned['current_dpd_state'].map(dpd_state_labels)

print("--- Current DPD state distribution (live loans) ---")
print(df_live_cleaned['current_dpd_label'].value_counts(dropna=False))

In [ ]:
# ============================================================
# CELL — Staging-specific override: reclassify Grace Period as early risk
# ============================================================
df_live_cleaned['current_dpd_state_for_staging'] = df_live_cleaned['current_dpd_state']
df_live_cleaned.loc[
    df_live_cleaned['loan_status'] == 'In Grace Period', 'current_dpd_state_for_staging'
] = 1  # treat as equivalent to 1-30 DPD for staging purposes only

df_live_cleaned['current_dpd_label_for_staging'] = df_live_cleaned['current_dpd_state_for_staging'].map(dpd_state_labels)

print("--- DPD state distribution FOR STAGING (Grace Period reclassified) ---")
print(df_live_cleaned['current_dpd_label_for_staging'].value_counts(dropna=False))

### Rebuilding Stage Assignment with the Corrected DPD Signal

The first staging attempt used `df_cleaned`, which only contains resolved
loans (Fully Paid, Charged Off, Default) -- a deliberate Week 1 filtering
choice for clean PD/LGD model training. Because of that, no loan in
`df_cleaned` was ever in an active delinquency state (1-30, 31-60, or 61-90
DPD), so the 30+ DPD staging trigger could never fire from real payment
behavior. Stage 2 in that first attempt was driven entirely by the PD-ratio
trigger, not genuine delinquency signal.

To fix this, live (still-active) loans were pulled from the original,
unfiltered dataset -- specifically `Current`, `In Grace Period`, and
`Late (16-30 days)` status loans, the only live statuses present in this
data snapshot in meaningful volume. These were scored with the existing
calibrated LightGBM PD model and assigned a real DPD state.

One additional correction was needed: `assign_state()` (built in Week 1 for
the transition matrix) classifies `In Grace Period` as equivalent to
`Current`, a defensible choice given grace period's contractual definition,
but one that hides an early risk signal relevant to staging. Rather than
modify `assign_state()` itself -- which would silently alter the already-
reported Week 1 transition matrix results -- a staging-specific override
reclassifies grace-period loans as equivalent to 1-30 DPD, used only in this
section.

The cell below rebuilds `staging_df` on this corrected foundation and checks
how many Stage 2 loans are now driven by real DPD signal versus the PD-ratio
fallback, to confirm the fix is actually doing meaningful work rather than
just changing labels without changing outcomes.

In [ ]:
# ============================================================
# CELL — Rebuild staging using the corrected DPD signal
# ============================================================
PD_DOUBLING_THRESHOLD = 2.0
DPD_TRIGGER_DAYS = 30

grade_avg_pd_live = current_pd_live.groupby(df_live_cleaned['grade']).transform('mean')
pd_ratio_live = current_pd_live / grade_avg_pd_live

def assign_stage(row_pd_ratio, dpd_state):
    if dpd_state == 4:                          # Default
        return 3
    if dpd_state in (1, 2, 3):                   # 1-30, 31-60, or 61-90 DPD (incl. reclassified Grace Period)
        return 2
    if row_pd_ratio >= PD_DOUBLING_THRESHOLD:     # PD significantly above grade norm
        return 2
    return 1

staging_df = pd.DataFrame({
    'pd_ratio': pd_ratio_live,
    'dpd_state': df_live_cleaned['current_dpd_state_for_staging'],
    'current_pd': current_pd_live,
}, index=df_live_cleaned.index).dropna()

staging_df['stage'] = staging_df.apply(
    lambda r: assign_stage(r['pd_ratio'], r['dpd_state']), axis=1
)

print("--- Stage distribution (live loans, corrected DPD signal) ---")
print(staging_df['stage'].value_counts().sort_index())
print(f"\nTotal live loans staged: {len(staging_df):,}")

print("\n--- Sanity check: how many Stage 2 loans came from DPD vs PD-ratio trigger? ---")
dpd_triggered = ((staging_df['dpd_state'].isin([1, 2, 3])) & (staging_df['stage'] == 2)).sum()
pdratio_triggered = ((~staging_df['dpd_state'].isin([1, 2, 3, 4])) & (staging_df['stage'] == 2)).sum()
print(f"Stage 2 via DPD trigger: {dpd_triggered:,}")
print(f"Stage 2 via PD-ratio trigger only: {pdratio_triggered:,}")

print("\n--- Stage distribution by DPD state (cross-check) ---")
print(pd.crosstab(staging_df['dpd_state'], staging_df['stage']))

### Computing ECL by Stage

In [ ]:
# ============================================================
# CELL — Build a forward-looking LGD model (origination/current
# features only), separate from the Week 1 realized-LGD model
# ============================================================
# The Week 1 LGD model used post-default features (loan_age_at_default,
# payment_ratio, recoveries_ratio, months_paid_ratio) -- correct for
# measuring REALIZED loss on already-defaulted loans, but inapplicable to
# live loans where default hasn't happened yet. This rebuilds a LGD model
# using only features known at any point during a loan's active life.

forward_lgd_features = [
    'int_rate', 'fico_range_low', 'dti', 'loan_amnt', 'annual_inc',
    'revol_util', 'loan_to_income', 'mort_acc', 'has_pub_rec',
    'grade_woe', 'home_ownership_woe'
]

# Training data: same defaulted-loan population Week 1 used for LGD,
# just restricted to forward-looking features and the actual realized
# LGD outcome as the target.
lgd_train_df = df_cleaned.loc[el_df.index][forward_lgd_features + []].copy()
lgd_train_df['realized_lgd'] = el_df['lgd']  # adjust column name if different
lgd_train_df = lgd_train_df.dropna()

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    lgd_train_df[forward_lgd_features], lgd_train_df['realized_lgd'],
    test_size=0.2, random_state=42
)

scaler_forward_lgd = StandardScaler()
X_train_scaled = scaler_forward_lgd.fit_transform(X_train)
X_test_scaled = scaler_forward_lgd.transform(X_test)

forward_lgd_model = GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)
forward_lgd_model.fit(X_train_scaled, y_train)

test_preds = forward_lgd_model.predict(X_test_scaled)
test_preds_clipped = np.clip(test_preds, 0, 1)
rmse = np.sqrt(mean_squared_error(y_test, test_preds_clipped))

print(f"Forward-looking LGD model RMSE: {rmse:.4f}")
print(f"Predicted LGD range on test set: {test_preds_clipped.min():.3f} to {test_preds_clipped.max():.3f}")
print(f"Predicted LGD mean on test set: {test_preds_clipped.mean():.3f}")

In [ ]:
# ============================================================
# CELL — Re-score Stage 1 ECL using the forward-looking LGD model
# ============================================================
stage1_mask = staging_df['stage'] == 1
stage1_loans = staging_df[stage1_mask].index

lgd_input_stage1 = df_live_cleaned.loc[stage1_loans, forward_lgd_features].dropna()
lgd_scaled_stage1 = scaler_forward_lgd.transform(lgd_input_stage1)
predicted_lgd_stage1 = pd.Series(
    np.clip(forward_lgd_model.predict(lgd_scaled_stage1), 0, 1),
    index=lgd_input_stage1.index
)

# EAD: current outstanding balance if out_prncp is available, else loan_amnt as a fallback
if 'out_prncp' in df_live_cleaned.columns:
    ead_stage1 = df_live_cleaned.loc[stage1_loans, 'out_prncp']
else:
    ead_stage1 = df_live_cleaned.loc[stage1_loans, 'loan_amnt']
    print("Using loan_amnt as EAD proxy -- out_prncp not available.")

common_idx_1 = stage1_loans.intersection(predicted_lgd_stage1.index).intersection(ead_stage1.index)

ecl_stage1 = pd.Series(index=staging_df.index, dtype=float)
ecl_stage1[common_idx_1] = (
    staging_df.loc[common_idx_1, 'current_pd'] *
    predicted_lgd_stage1.reindex(common_idx_1) *
    ead_stage1.reindex(common_idx_1)
)

print(f"Stage 1 loans: {stage1_mask.sum():,}")
print(f"Stage 1 loans with complete data for ECL: {len(common_idx_1):,}")
print(f"Stage 1 mean predicted LGD: {predicted_lgd_stage1.mean():.4f}")
print(f"Stage 1 total 12-month ECL: ${ecl_stage1[common_idx_1].sum():,.0f}")

In [ ]:
# ============================================================
# CELL — Re-score Stage 2 ECL using the forward-looking LGD model
# ============================================================
stage2_mask = staging_df['stage'] == 2
stage2_loans = staging_df[stage2_mask].index

cox_input_stage2 = df_live_cleaned.loc[stage2_loans, cox_features].dropna()
remaining_term_stage2 = df_live_cleaned.loc[cox_input_stage2.index, 'term'].astype(str).str.extract(r'(\d+)').astype(float).squeeze()
remaining_term_stage2 = remaining_term_stage2.fillna(36)

print(f"Computing lifetime survival curves for {len(cox_input_stage2):,} Stage 2 loans...")
sf_stage2 = cph.predict_survival_function(cox_input_stage2)

lifetime_pd_stage2 = pd.Series(index=cox_input_stage2.index, dtype=float)
for loan_id in cox_input_stage2.index:
    term_months = remaining_term_stage2.loc[loan_id]
    nearest_idx = sf_stage2.index.get_indexer([term_months], method='nearest')[0]
    survival_prob = sf_stage2.loc[sf_stage2.index[nearest_idx], loan_id]
    lifetime_pd_stage2[loan_id] = 1 - survival_prob

lgd_input_stage2 = df_live_cleaned.loc[cox_input_stage2.index, forward_lgd_features].dropna()
lgd_scaled_stage2 = scaler_forward_lgd.transform(lgd_input_stage2)
predicted_lgd_stage2 = pd.Series(
    np.clip(forward_lgd_model.predict(lgd_scaled_stage2), 0, 1),
    index=lgd_input_stage2.index
)

if 'out_prncp' in df_live_cleaned.columns:
    ead_stage2 = df_live_cleaned.loc[cox_input_stage2.index, 'out_prncp']
else:
    ead_stage2 = df_live_cleaned.loc[cox_input_stage2.index, 'loan_amnt']

common_idx_2 = lifetime_pd_stage2.index.intersection(predicted_lgd_stage2.index).intersection(ead_stage2.index)

ecl_stage2 = pd.Series(index=staging_df.index, dtype=float)
ecl_stage2[common_idx_2] = (
    lifetime_pd_stage2.reindex(common_idx_2) *
    predicted_lgd_stage2.reindex(common_idx_2) *
    ead_stage2.reindex(common_idx_2)
)

print(f"Stage 2 loans: {stage2_mask.sum():,}")
print(f"Stage 2 loans with complete data for ECL: {len(common_idx_2):,}")
print(f"Stage 2 mean lifetime PD: {lifetime_pd_stage2.mean():.4f}")
print(f"Stage 2 mean predicted LGD: {predicted_lgd_stage2.mean():.4f}")
print(f"Stage 2 total lifetime ECL: ${ecl_stage2[common_idx_2].sum():,.0f}")

### Portfolio Provisions Summary

In [ ]:
# ============================================================
# CELL — Combine Stage 1 + Stage 2 into the portfolio provisions summary
# ============================================================
# NOTE: EAD uses original loan_amnt as a proxy for outstanding balance,
# since 'out_prncp' wasn't pulled into df_live_cleaned. This overstates
# exposure for loans that have already paid down principal, and therefore
# overstates total ECL somewhat. Documented here as a known limitation
# rather than corrected, to keep momentum on the Basel III work.

total_ecl_live = ecl_stage1.fillna(0) + ecl_stage2.fillna(0)

provisions_df = pd.DataFrame({
    'stage': staging_df['stage'],
    'ecl': total_ecl_live,
})

print("--- Portfolio Provisions Summary (IFRS 9, live loans) ---")
summary = provisions_df.groupby('stage').agg(
    loan_count=('ecl', 'count'),
    total_ecl=('ecl', 'sum'),
)
summary['avg_ecl_per_loan'] = summary['total_ecl'] / summary['loan_count']
print(summary)

print(f"\nTotal live-portfolio IFRS 9 provisions: ${provisions_df['ecl'].sum():,.0f}")
print(f"\nKNOWN LIMITATION: EAD uses original loan_amnt, not current outstanding "
      f"balance (out_prncp unavailable). This overstates exposure for partially "
      f"paid-down loans, so the totals above are an upper-bound estimate, not a "
      f"precise figure.")

In [ ]:
# ============================================================
# CELL — Visualize stage distribution and provision contribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

stage_counts = staging_df['stage'].value_counts().sort_index()
stage_labels = {1: 'Stage 1\n(Performing)', 2: 'Stage 2\n(Underperforming)'}
axes[0].bar([stage_labels[s] for s in stage_counts.index], stage_counts.values,
            color=['#2ecc71', '#f39c12'])
axes[0].set_title('Live Loan Count by Stage')
axes[0].set_ylabel('Number of Loans')

stage_ecl = provisions_df.groupby('stage')['ecl'].sum()
axes[1].bar([stage_labels[s] for s in stage_ecl.index], stage_ecl.values,
            color=['#2ecc71', '#f39c12'])
axes[1].set_title('Total Provisions ($) by Stage (upper-bound, see EAD note)')
axes[1].set_ylabel('Provisions ($)')

plt.tight_layout()
plt.show()

Result: Loan count: Stage 1 (866,689) completely dwarfs Stage 2 (~22,225) — about 97.5% of live loans are performing fine, 2.5% are flagged as deteriorating. That's a realistic, healthy-looking split for a real loan book.
Provisions: Stage 1's bar is still much taller in absolute dollars ($1.36B vs Stage 2's $83M) simply because Stage 1 has 39x more loans — even a small average provision per loan adds up to a big total across nearly 867K loans. Stage 2's much smaller loan count still produces a non-trivial $83M, which reflects the much higher per-loan risk (36% PD vs whatever Stage 1's average is) concentrated in a tiny fraction of the book.
This is actually a nice, clean illustration of the exact mechanic IFRS 9 staging exists to capture — a small flagged subset carrying disproportionate risk-per-loan, even though it doesn't dominate by total dollars in this particular snapshot. Worth keeping that framing for the writeup.